# Data Preprocessing Module
____

## Overview
This notebook demonstrates the complete data preprocessing pipeline for the Grievance Redressal System. The workflow includes data acquisition, filtering, text normalization, and preparation for downstream machine learning tasks.

In [27]:
import numpy as np
import pandas as pd

## Data Loading and Exploration
____

### Initial Data Acquisition
Loading the grievance dataset and performing basic exploratory data analysis to understand the data structure and characteristics.

In [28]:
df = pd.read_csv('../data/grievance_dataset.csv')

In [29]:
df.shape

(156600, 8)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156600 entries, 0 to 156599
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   _id             156600 non-null  object
 1   org_code        156600 non-null  object
 2   recvd_date      156600 non-null  object
 3   closing_date    156600 non-null  object
 4   sex             156600 non-null  object
 5   state           156600 non-null  object
 6   grievance_text  156600 non-null  object
 7   remarks_text    156600 non-null  object
dtypes: object(8)
memory usage: 9.6+ MB


In [31]:
df.sample(10)

,_id,org_code,recvd_date,closing_date,sex,state,grievance_text,remarks_text
156218,DEABD/E/2023/0008605,DEABD,2023-01-31 20:01:45.810000+00:00,2023-02-08 00:00:00+00:00,Female,UP,Financial Services (Banking Division) >> Misbe...,Duplicate of CPGRAMS Complaint No.XEXBX/X/X0X...
110914,MOCOP/E/2023/0000703,MOCOP,2023-01-22 21:37:11.493000+00:00,2023-01-27 00:00:00+00:00,Male,MP,Cooperation >> Central Registrar and Cooperati...,Reply Attached.
51671,DOURD/E/2023/0000566,DOURD,2023-01-10 13:38:45.143000+00:00,2023-01-16 00:00:00+00:00,Male,GJ,Housing and Urban Affairs >> PMAY - URBAN/ Hou...,"Respected Sir/Madam,\r\nThe release of subsid..."
143642,PMOPG/E/2023/0025722,PMOPG,2023-01-29 00:00:00+00:00,2023-03-17 00:00:00+00:00,Male,KA,Greetings to Honorable Prime Minister Shri Nar...,"With reference to your grievance, it is here..."
103952,GOVUC/E/2023/0000173,GOVUC,2023-01-21 00:00:00+00:00,2023-01-24 00:00:00+00:00,Male,UP,Establishment did miss lead to Government in s...,No remarks provided
100989,DPOST/E/2023/0002550,DPOST,2023-01-20 08:57:39.783000+00:00,2023-01-31 00:00:00+00:00,Male,MH,Posts >> Others/Misc.\r\n---------------------...,"Sir,\r\nAs per SPM Kothrud SO, the mobile num..."
152633,MORLY/E/2023/0002683,MORLY,2023-01-31 00:00:00+00:00,2023-02-03 00:00:00+00:00,Male,BR,Mai Sagar kumar LPG Q RPH Working on Loan AZ M...,Replied with enclosed document please.
28479,PMOPG/D/2023/0006918,PMOPG,2023-01-06 00:00:00+00:00,2023-01-11 00:00:00+00:00,Male,DN,FRANK1,No remarks provided
97035,DHLTH/E/2023/0000749,DHLTH,2023-01-19 16:22:18.303000+00:00,2023-03-22 00:00:00+00:00,Male,UP,Health & Family Welfare >> Health Schemes >> P...,Please refer the above attachment. The case m...
117949,PMOPG/D/2023/0019126,PMOPG,2023-01-24 00:00:00+00:00,2023-03-03 00:00:00+00:00,Male,UP,TV,No remarks provided


In [32]:
df = df.set_index('_id')

&nbsp;

##  Data Filtering and Quality Assurance
____

### Data Preprocessing Criteria
This section applies rigorous filtering criteria to ensure data quality:
- **Content Validation**: Retains only records containing ">>" markers
- **Language Filter**: Ensures text contains only ASCII characters (English language)
- **Length Requirement**: Enforces minimum text length of 100 characters
- **Status Filter**: Excludes records with "pending" organization codes

In [33]:
# Ensure string type
df['grievance_text'] = df['grievance_text'].astype(str)

# ASCII pattern (allows English + numbers + common symbols)
pattern = r"^[\x00-\x7F]+$"

# Apply all filters
filtered_df = df[
    (df['grievance_text'].str.contains(">>", na=False)) &        # contains >>
    (df['grievance_text'].str.fullmatch(pattern, na=False)) &    # only ASCII (English)
    (df['grievance_text'].str.len() > 100) &                     # len > 100
    (df['org_code'].str.lower() != "pending")                    # exclude 'pending'
]

print(f"Done! Extracted {len(filtered_df)} rows.")

Done! Extracted 39971 rows.


In [34]:
filtered_df.shape

(39971, 7)

In [35]:
filtered_df['org_code'].nunique()

88

In [36]:
filtered_df['org_code'].value_counts()

org_code
MOLBR    7939
DEABD    4542
CBODT    4306
DPOST    3035
DOTEL    2401
         ... 
DOFPI       5
DONER       5
DOSIR       5
DFSHR       4
DSPAC       3
Name: count, Length: 88, dtype: int64

&nbsp;

## Text Normalization and Cleaning
____

### Data Preparation
Creating a copy of the filtered dataset for comprehensive text preprocessing operations.

In [37]:
filtered_grievance = filtered_df.copy()

In [38]:
filtered_grievance.sample(10)

,org_code,recvd_date,closing_date,sex,state,grievance_text,remarks_text
_id,,,,,,,
MOLBR/E/2023/0002342,MOLBR,2023-01-06 14:11:16.510000+00:00,2023-01-19 00:00:00+00:00,Male,GJ,Labour and Employment >> Pension HQ\r\n\r\nNam...,It is to inform that please update your mobil...
MORTH/E/2023/0000946,MORTH,2023-01-18 07:49:39.373000+00:00,2023-02-03 00:00:00+00:00,Male,TS,Road Transport and Highways >> Policiy Related...,Reply copy attached.
DHLTH/E/2023/0000391,DHLTH,2023-01-10 18:52:16.160000+00:00,2023-01-13 00:00:00+00:00,Male,BR,Health & Family Welfare >> Public Health >> Ot...,No remarks provided
DPOST/E/2023/0001228,DPOST,2023-01-10 14:16:40.990000+00:00,2023-01-12 00:00:00+00:00,Female,GJ,Posts >> Others/Misc.\r\n---------------------...,It is to intimate that article number.X-X-X-X...
DHLTH/E/2023/0000349,DHLTH,2023-01-09 20:22:29.647000+00:00,2023-03-09 00:00:00+00:00,Female,GJ,Health & Family Welfare >> Health Schemes >> N...,જરૂરી કાર્યવાહી અર્થે સંબંધિત જિલ્લા વિકાસ અધ...
DEPOJ/E/2023/0001292,DEPOJ,2023-01-26 09:49:08.807000+00:00,2023-05-05 00:00:00+00:00,Male,GJ,Justice >> e-Court >> Video conferencing in co...,આ કામના અરજદાશ્રીએ તાપી જિલ્લાના સોનગઢ તાલુકાન...
DEAPR/E/2023/0000135,DEAPR,2023-01-12 18:59:08.620000+00:00,2023-01-16 00:00:00+00:00,Male,GJ,Financial Services (Pension Reforms) >> Others...,Please find attached herewith copy of reply vi...
MOLBR/E/2023/0008582,MOLBR,2023-01-24 21:53:14.760000+00:00,2023-01-25 00:00:00+00:00,Male,JH,Labour and Employment >> Transfer related issu...,As per requirement of Complainant Annexure K ...
MORTH/E/2023/0000028,MORTH,2023-01-02 10:50:33.417000+00:00,2023-01-06 00:00:00+00:00,Male,TN,Road Transport and Highways >> Construction & ...,Reply copy attached.



### Text Processing Pipeline
The following transformations are sequentially applied to the grievance text:

#### Case Normalization

In [39]:
filtered_grievance['grievance_text'] = filtered_grievance['grievance_text'].astype(str)
filtered_grievance['grievance_text_processed'] = filtered_grievance['grievance_text']

In [40]:
filtered_grievance['grievance_text_processed'] = filtered_grievance['grievance_text_processed'].str.lower()

#### HTML Tag Removal

In [41]:
import re

def remove_html_tags(text):
    if pd.isna(text):
        return text
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

filtered_grievance['grievance_text_processed'] = filtered_grievance['grievance_text_processed'].apply(remove_html_tags)

#### URL and Hyperlink Removal

In [42]:
import re

def remove_url(text):
    if pd.isna(text): 
        return text
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

filtered_grievance['grievance_text_processed'] = filtered_grievance['grievance_text_processed'].apply(remove_url)

#### Punctuation Removal

In [43]:
import string
exclude = string.punctuation

def remove_punc_better(text):
    if pd.isna(text): 
        return text
    return text.translate(str.maketrans('', '', exclude))

filtered_grievance['grievance_text_processed'] = filtered_grievance['grievance_text_processed'].apply(remove_punc_better)

#### Stopword Removal
Eliminating common English words that do not contribute meaningful information to the analysis.

In [44]:
from nltk.corpus import stopwords
import nltk

# Download stopwords once
# nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    if pd.isna(text):
        return text
    # Split and filter stopwords
    words = [word for word in text.split() if word not in stop_words]
    return " ".join(words)

filtered_grievance['grievance_text_processed'] = filtered_grievance['grievance_text_processed'].apply(remove_stopwords)

In [45]:
filtered_grievance.sample(10)

,org_code,recvd_date,closing_date,sex,state,grievance_text,remarks_text,grievance_text_processed
_id,,,,,,,,
DPOST/E/2023/0004195,DPOST,2023-01-31 18:31:12.647000+00:00,2023-02-06 00:00:00+00:00,Male,TN,Posts >> Others/Misc.\r\n---------------------...,Reply enclosed.,posts othersmisc earlier complaint issue solve...
DOAAC/E/2023/0000869,DOAAC,2023-01-05 09:13:18.567000+00:00,2023-02-06 00:00:00+00:00,Male,UP,Agriculture and Farmers Welfare >> PMKISAN rel...,No remarks provided,agriculture farmers welfare pmkisan related is...
DEABD/E/2023/0006420,DEABD,2023-01-21 17:20:40.193000+00:00,2023-02-09 00:00:00+00:00,Male,UP,Financial Services (Banking Division) >> Credi...,Dear Sir/Madam\r\n\r\nWith reference to capti...,financial services banking division creditdebi...
DOSEL/E/2023/0000436,DOSEL,2023-01-28 20:20:59.967000+00:00,2023-02-08 00:00:00+00:00,Male,CH,School Education and Literacy >> Others/Misc\r...,Private Coaching Centre does not come under t...,school education literacy othersmisc dear sir ...
CBODT/E/2023/0004687,CBODT,2023-01-23 18:27:44.917000+00:00,2023-02-02 00:00:00+00:00,Female,GJ,Central Board of Direct Taxes (Income Tax) >> ...,"Dear Sir/Madam, As seen from the CPC portal f...",central board direct taxes income tax direct t...
MOLBR/E/2023/0007818,MOLBR,2023-01-22 20:56:32.680000+00:00,2023-02-07 00:00:00+00:00,Male,JH,Labour and Employment >> PF Withdrawal >> Dela...,Transfer-in amount has been credited in prese...,labour employment pf withdrawal delay final se...
DEABD/E/2023/0006561,DEABD,2023-01-22 13:56:44.580000+00:00,2023-01-27 00:00:00+00:00,Male,HR,Financial Services (Banking Division) >> Credi...,Dear Sir/ Madam\r\nThis is with reference to C...,financial services banking division creditdebi...
AYUSH/E/2023/0000033,AYUSH,2023-01-19 22:24:44.307000+00:00,2023-02-21 00:00:00+00:00,Female,KA,Ayush >> Miscellaneous >> All field Organizati...,This issue was verified from NTA. The Score C...,ayush miscellaneous field organizations moa re...
CBOEC/E/2023/0000257,CBOEC,2023-01-11 19:58:45.937000+00:00,2023-02-21 00:00:00+00:00,Male,AP,Central Board of Excise and Customs >> Indirec...,Reply uploaded in the CPGRAM portal on 21.02....,central board excise customs indirect taxes re...


&nbsp;

## Data Deduplication and Finalization
____

### Duplicate Removal
Eliminating duplicate grievance records while preserving the first occurrence of each unique grievance text. This ensures that the downstream models are not biased by repeated entries.

In [46]:
grievance_domain_data = filtered_grievance.copy()

# Display original shape
print(f"Original shape: {grievance_domain_data.shape}")

# Remove rows with duplicate grievance_text (keeps first occurrence)
df_deduplicated = grievance_domain_data.drop_duplicates(subset=['grievance_text'], keep='first')

# Display new shape
print(f"After deduplication shape: {df_deduplicated.shape}")
print(f"Duplicates removed: {grievance_domain_data.shape[0] - df_deduplicated.shape[0]}")

Original shape: (39971, 8)
After deduplication shape: (37854, 8)
Duplicates removed: 2117


In [47]:
df_deduplicated.to_csv("../data/grievance_dataset_domain_classification.csv")